# Scratch Work for Flow Matching model 

In [1]:
from ase.io import read
from ase import Atoms
from ase.visualize import view

import torch 
from torch import optim
from torch import nn
from torch_geometric.data import Data
from torch import Tensor

from models.gemnet.gemnet_flow import GemNetFlow
from models.gemnet.gemnet_base import GemNetOC
from ase.io import read


/opt/anaconda3/envs/cfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
contcar = read("POSCAR.vasp")
poscar = read("POSCAR0.vasp")
# Concvert atomic positions to data object
positions = torch.tensor(contcar.get_positions(), dtype=torch.float32)
atomic_numbers = torch.tensor(contcar.get_atomic_numbers(), dtype=torch.long)

## Compile GemNet for flow matching

In [3]:
# GemNet-OC params
model_cfg = {
    "num_targets": 1,
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 4,
    "emb_size_atom": 256,
    "emb_size_edge": 512,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "num_atom_emb_layers": 0,
    "num_global_out_layers": 2,
    "flow_matching": True,
    "dummy": True,
    "use_pbc": True,
    "scale_backprop_forces": False,
    "cutoff": 12.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 30,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "enforce_max_neighbors_strictly": True,
    "rbf": {"name": "gaussian"},
    "rbf_spherical": None,
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "forces_coupled": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": None,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "scale_basis": True,
}


In [4]:
GemNet = GemNetOC(**model_cfg)

# Count total number of parameters
trainable_params = sum(p.numel() for p in GemNet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 37815808


## Testing GemNet-OC for energy/force regression

In [5]:
optimizer = optim.Adam(GemNet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(100): 
  
        # Create batch index (all atoms belong to structure 0)
        x1 = positions 
        batch = torch.zeros(len(x1), dtype=torch.long)

        # Build data objects
        data = Data(pos=x1, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        output = GemNet(data)

        # Compute loss 
        force = torch.tensor([[0, 0, 1], [0, 0, -1]],  dtype=torch.float32)
        loss = loss_fn(output, force)

        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.0001:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    return output




## Trainer

In [6]:
GemFlow = GemNetFlow(**model_cfg)

In [ ]:
optimizer = optim.Adam(GemFlow.parameters(), 1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(1000): 
        
        if epoch == 0: 
            x1 = positions 
            x0 = torch.randn_like(x1) 
            t = torch.rand(1)
            xt = (1 - t) * x0 + t * x1
        
        batch = torch.zeros(len(xt), dtype=torch.long)

        # Build data objects
        data = Data(pos=xt, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.t = t
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        u_theta = GemFlow(data)
    
        # Compute loss 
        loss = loss_fn(u_theta, x1 - x0)
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.01:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
train()

Loss at epoch 0: 0.9004583358764648
Loss at epoch 1: 998.6508178710938
Loss at epoch 2: 1334.506103515625
Loss at epoch 3: 0.9334464073181152
Loss at epoch 4: 3.370925188064575
Loss at epoch 5: 0.7042193412780762
Loss at epoch 6: 0.7134634852409363
Loss at epoch 7: 0.6585052609443665
Loss at epoch 8: 0.5273453593254089
Loss at epoch 9: 39.610836029052734
Loss at epoch 10: 0.7238909602165222
Loss at epoch 11: 0.8110343813896179
Loss at epoch 12: 0.8199396133422852
Loss at epoch 13: 0.763916015625
Loss at epoch 14: 0.6314077377319336
Loss at epoch 15: 16227.8896484375
Loss at epoch 16: 0.6913105845451355
Loss at epoch 17: 0.7320221066474915
Loss at epoch 18: 0.7409920692443848
Loss at epoch 19: 0.7334234118461609
Loss at epoch 20: 0.714285671710968
Loss at epoch 21: 0.6846082806587219
Loss at epoch 22: 0.6442639231681824
Loss at epoch 23: 0.5938945412635803
Loss at epoch 24: 0.5368812084197998
Loss at epoch 25: 0.4817143976688385
Loss at epoch 26: 0.44425833225250244
Loss at epoch 27: 0.

KeyboardInterrupt: 

In [11]:
optimizer = optim.Adam(GemFlow.parameters(), 1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(100000): 
    
        x1 = positions 
        x0 = torch.randn_like(x1) 
        t = torch.rand(1)
        xt = (1 - t) * x0 + t * x1
        
        batch = torch.zeros(len(xt), dtype=torch.long)

        # Build data objects
        data = Data(pos=xt, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.t = t
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        u_theta = GemFlow(data)
    
        # Compute loss 
        loss = loss_fn(u_theta, x1 - x0)
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.01:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
train()

Loss at epoch 0: 0.7420021891593933
Loss at epoch 1: 23241.609375
Loss at epoch 2: 4225.07861328125
Loss at epoch 3: 37.80196762084961
Loss at epoch 4: 11.095488548278809
Loss at epoch 5: 501.52978515625
Loss at epoch 6: 109.96184539794922
Loss at epoch 7: 69.58100891113281
Loss at epoch 8: 1.9438802003860474
Loss at epoch 9: 341.1330261230469
Loss at epoch 10: 8.719185829162598
Loss at epoch 11: 3.0007846355438232
Loss at epoch 12: 9.506187438964844
Loss at epoch 13: 1.542199730873108
Loss at epoch 14: 13.917476654052734
Loss at epoch 15: 2.6316492557525635
Loss at epoch 16: 1.6338542699813843
Loss at epoch 17: 0.4951219856739044
Loss at epoch 18: 1.3498767614364624
Loss at epoch 19: 2.892780065536499
Loss at epoch 20: 0.7460228800773621
Loss at epoch 21: 1.1546348333358765
Loss at epoch 22: 7.003145694732666
Loss at epoch 23: 2.4793975353240967
Loss at epoch 24: 1.1756564378738403
Loss at epoch 25: 0.978499710559845
Loss at epoch 26: 0.3078407347202301
Loss at epoch 27: 0.90598225593

KeyboardInterrupt: 

In [ ]:
# Create batch index (all atoms belong to structure 0)
contcar = read("POSCAR.vasp")
positions = torch.tensor(contcar.get_positions(), dtype=torch.float32)
x1 = positions 
x0 = torch.randn_like(x1)
x0_p = torch.tensor([[0, 0, 1], [0, 0, -1]])
print(x1)
print(x0)
print(x0_p)

tensor([[ 0.0000,  0.0000,  0.3686],
        [ 0.0000,  0.0000, -0.3686]])
tensor([[-0.2181,  0.8389,  0.8817],
        [-0.2634, -0.3204, -0.1327]])
tensor([[ 0,  0,  1],
        [ 0,  0, -1]])


## Sampler